In [16]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

In [17]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = housing.data        # shape (20640, 8)
y = housing.target.reshape(-1, 1)  # shape (20640, 1)

print(housing.feature_names)  #

['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val   = torch.tensor(X_val,   dtype=torch.float32)
y_val   = torch.tensor(y_val,   dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)


In [19]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

torch.Size([14448, 8])
torch.Size([3096, 8])
torch.Size([3096, 8])


In [20]:
class LinearRegressionModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel(n_features=8)
print(model)

LinearRegressionModel(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)


In [21]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [22]:
train_losses = []
epochs = 500

for epoch in range(epochs):
    model.train()
    
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.0f}")

Epoch 0 | Loss: 4
Epoch 50 | Loss: 1
Epoch 100 | Loss: 1
Epoch 150 | Loss: 1
Epoch 200 | Loss: 1
Epoch 250 | Loss: 1
Epoch 300 | Loss: 1
Epoch 350 | Loss: 1
Epoch 400 | Loss: 1
Epoch 450 | Loss: 1


In [23]:
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test)

In [24]:
# RMSE — root mean squared error
rmse = torch.sqrt(torch.mean((y_pred_test - y_test) ** 2)).item()

# R² — how much variance the model explains
ss_res = torch.sum((y_test - y_pred_test) ** 2)
ss_tot = torch.sum((y_test - y_test.mean()) ** 2)
r2 = (1 - ss_res / ss_tot).item()

In [25]:
print(f"Test RMSE: ${rmse * 100000:,.0f}")
print(f"Test R²: {r2:.4f}")

Test RMSE: $72,727
Test R²: 0.6000


In [26]:
torch.save(model.state_dict(), "model_california.pth")

import pickle
with open("scaler_california.pkl", "wb") as f:
    pickle.dump(scaler, f)

np.save("y_all_california.npy", y)